In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import joblib

model_path = "/content/drive/MyDrive/temperature_humidity_model.pkl"

model = joblib.load(model_path)

print("ML model loaded successfully!")

ML model loaded successfully!


In [4]:
!pip install paho-mqtt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.2 MB/s eta 0:00:00


In [5]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.9 MB/s eta 0:00:00


In [6]:
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [10]:
from google.colab import userdata
from groq import Groq

# Get Groq API key from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# Check whether the key was loaded
if GROQ_API_KEY:
    print("GROQ_API_KEY loaded successfully")
else:
    print("GROQ_API_KEY was not found")

# Create Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq LLM client initialized successfully!")

GROQ_API_KEY loaded successfully
Groq LLM client initialized successfully!


In [13]:
# ============================================================
# REAL-TIME ML + RAG + LLM + SQLITE + MQTT
# Temperature & Humidity Fault Detection
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    groq \
    paho-mqtt \
    openpyxl


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import json
import sqlite3
import joblib
import os
import paho.mqtt.client as mqtt

from datetime import datetime
from zoneinfo import ZoneInfo

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from groq import Groq

from google.colab import userdata
from google.colab import drive
from google.colab import files


# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

print("Google Drive mounted successfully!")



print("Model path:", MODEL_PATH)


# ============================================================
# 5. LOAD MAINTENANCE MANUAL
# ============================================================

PDF_PATH = "/content/drive/MyDrive/temperature_humidity_maintenance_manual.pdf"

print("\nLoading maintenance manual...")

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"Maintenance manual not found at:\n{PDF_PATH}"
    )

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("Maintenance manual loaded successfully!")
print("Number of pages:", len(documents))


# ============================================================
# 6. SPLIT PDF INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("PDF split into", len(chunks), "chunks")


# ============================================================
# 7. CREATE EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")


# ============================================================
# 8. CREATE FAISS VECTOR DATABASE
# ============================================================

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")


# ============================================================
# 9. LOAD GROQ API KEY
# ============================================================

print("\nLoading Groq API key...")

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found in Colab Secrets."
    )

groq_client = Groq(
    api_key=GROQ_API_KEY
)


# ============================================================
# IMPORTANT:
# GROQ MODEL
# ============================================================

LLM_MODEL = "openai/gpt-oss-20b"

print("Groq LLM initialized successfully!")
print("Using Groq model:", LLM_MODEL)


# ============================================================
# 10. CREATE SQLITE DATABASE
# ============================================================

DB_PATH = "temperature_monitoring.db"

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sensor_data (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    date TEXT,

    time TEXT,

    temperature REAL,

    humidity REAL,

    prediction TEXT,

    fault_description TEXT,

    recommendation TEXT

)
""")

conn.commit()
conn.close()

print("SQLite database created successfully!")


# ============================================================
# 11. MQTT SETTINGS
# ============================================================

BROKER = "broker.hivemq.com"

PORT = 1883

TOPIC = "power/fault"


# ============================================================
# 12. CREATE RESULTS DATAFRAME
# ============================================================

results = pd.DataFrame(
    columns=[
        "Date",
        "Time",
        "Temperature",
        "Humidity",
        "Prediction",
        "Fault Description",
        "Recommendation"
    ]
)


# ============================================================
# 13. MQTT CONNECT CALLBACK
# ============================================================

def on_connect(client, userdata, flags, reason_code, properties):

    if reason_code == 0:

        print("\nConnected to MQTT broker")

        client.subscribe(TOPIC)

        print("Subscribed to:", TOPIC)

    else:

        print(
            "MQTT connection failed. Reason code:",
            reason_code
        )


# ============================================================
# 14. MQTT MESSAGE CALLBACK
# ============================================================

def on_message(client, userdata, msg):

    global results

    try:

        # ====================================================
        # STEP 1: RECEIVE MQTT MESSAGE
        # ====================================================

        message = msg.payload.decode()

        print("\nReceived:", message)


        # ====================================================
        # STEP 2: CONVERT JSON
        # ====================================================

        sensor_data = json.loads(message)


        # ====================================================
        # STEP 3: GET TEMPERATURE AND HUMIDITY
        # ====================================================

        temperature = float(
            sensor_data["Temperature"]
        )

        humidity = float(
            sensor_data["Humidity"]
        )


        # ====================================================
        # STEP 4: PREPARE ML INPUT
        # ====================================================

        X_new = pd.DataFrame({

            "Temperature": [temperature],

            "Humidity": [humidity]

        })


        # ====================================================
        # STEP 5: ML INFERENCE
        # ====================================================

        prediction = model.predict(X_new)[0]

        prediction = str(prediction).strip()


        print("ML Prediction:", prediction)


        # ====================================================
        # STEP 6: CONVERT ML RESULT TO FAULT DESCRIPTION
        # ====================================================

        prediction_lower = prediction.lower()


        if prediction_lower == "healthy":

            fault_description = (
                "Machine is Healthy"
            )


        elif prediction_lower == "fault":

            fault_description = (
                "Temperature and Humidity "
                "Abnormality Detected"
            )


        else:

            fault_description = (
                "Unknown Machine Condition"
            )


        # ====================================================
        # STEP 7: RAG SEARCH
        # ====================================================

        try:

            query = (
                fault_description
                + " temperature humidity "
                + "sensor maintenance troubleshooting "
                + "corrective action"
            )

            retrieved_docs = vector_db.similarity_search(
                query,
                k=3
            )

            retrieved_context = "\n\n".join(
                doc.page_content
                for doc in retrieved_docs
            )

            print("RAG search completed successfully!")


        except Exception as rag_error:

            print(
                "RAG Error:",
                rag_error
            )

            retrieved_context = (
                "Relevant maintenance information "
                "could not be retrieved."
            )


        # ====================================================
        # STEP 8: CREATE LLM PROMPT
        # ====================================================

        prompt = f"""

You are an industrial maintenance assistant.

The real-time monitoring system has detected:

Temperature: {temperature} °C

Humidity: {humidity} %

ML Prediction: {prediction}

Fault Description: {fault_description}


Relevant information retrieved from the
temperature and humidity maintenance manual:

--------------------------------
{retrieved_context}
--------------------------------


Based ONLY on the retrieved maintenance
manual information, provide a maintenance
recommendation.

Include:

1. Possible Cause
2. Recommended Inspection
3. Corrective Action
4. Safety Precaution


Use simple and clear technical language.

Important:

The ML prediction is an indication and
not a confirmed physical machine fault.

Do not invent maintenance procedures that
are not supported by the maintenance manual.

"""


        # ====================================================
        # STEP 9: GET INDIA DATE AND TIME
        # ====================================================

        now = datetime.now(
            ZoneInfo("Asia/Kolkata")
        )

        date_value = now.strftime(
            "%Y-%m-%d"
        )

        time_value = now.strftime(
            "%H:%M:%S"
        )


        # ====================================================
        # STEP 10: SEND PROMPT TO GROQ
        # ====================================================

        try:

            response = groq_client.chat.completions.create(

                # IMPORTANT:
                # Do NOT use llama-3.3-70b-versatile here

                model=LLM_MODEL,

                messages=[

                    {
                        "role": "system",

                        "content":
                        "You are an industrial "
                        "maintenance assistant."
                    },

                    {
                        "role": "user",

                        "content": prompt
                    }

                ],

                temperature=0.2,

                max_completion_tokens=500
            )


            # =================================================
            # STEP 11: GET LLM RECOMMENDATION
            # =================================================

            recommendation = (
                response
                .choices[0]
                .message
                .content
            )

            print("Groq LLM recommendation generated successfully!")


        except Exception as llm_error:

            # =================================================
            # IMPORTANT:
            # DO NOT STOP THE PROGRAM IF GROQ FAILS
            # =================================================

            print(
                "\nGroq LLM Error:",
                llm_error
            )

            recommendation = (
                "LLM recommendation unavailable. "
                "Please inspect the temperature and "
                "humidity sensor, sensor wiring, "
                "and operating conditions."
            )


        # ====================================================
        # STEP 12: SAVE DATA TO SQLITE
        # ====================================================

        conn = sqlite3.connect(DB_PATH)

        cursor = conn.cursor()


        cursor.execute("""
        INSERT INTO sensor_data
        (
            date,
            time,
            temperature,
            humidity,
            prediction,
            fault_description,
            recommendation
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,

        (
            date_value,
            time_value,
            temperature,
            humidity,
            prediction,
            fault_description,
            recommendation
        ))


        conn.commit()

        conn.close()


        # ====================================================
        # STEP 13: SAVE TO DATAFRAME
        # ====================================================

        new_row = pd.DataFrame({

            "Date": [date_value],

            "Time": [time_value],

            "Temperature": [temperature],

            "Humidity": [humidity],

            "Prediction": [prediction],

            "Fault Description": [
                fault_description
            ],

            "Recommendation": [
                recommendation
            ]

        })


        results = pd.concat(
            [results, new_row],
            ignore_index=True
        )


        # ====================================================
        # STEP 14: DISPLAY COMPLETE RESULT
        # ====================================================

        print("\n")

        print("=" * 70)

        print(
            "       REAL-TIME ML + RAG + LLM RESULT"
        )

        print("=" * 70)


        print(
            "Date              :",
            date_value
        )


        print(
            "Time              :",
            time_value
        )


        print(
            "Temperature       :",
            temperature,
            "°C"
        )


        print(
            "Humidity          :",
            humidity,
            "%"
        )


        print(
            "ML Prediction     :",
            prediction
        )


        print(
            "Fault Description :",
            fault_description
        )


        print("-" * 70)

        print(
            "LLM MAINTENANCE RECOMMENDATION"
        )

        print("-" * 70)


        print(recommendation)


        print("-" * 70)

        print(
            "Data saved to SQLite successfully!"
        )

        print("=" * 70)


    except Exception as e:

        print("\n========================================")

        print("MQTT PROCESSING ERROR")

        print("========================================")

        print("Error:", e)

        print("========================================")


# ============================================================
# 15. CREATE MQTT CLIENT
# ============================================================

client = mqtt.Client(
    callback_api_version=
    mqtt.CallbackAPIVersion.VERSION2
)


# ============================================================
# 16. ASSIGN MQTT CALLBACKS
# ============================================================

client.on_connect = on_connect

client.on_message = on_message


# ============================================================
# 17. CONNECT TO MQTT BROKER
# ============================================================

print("\nConnecting to MQTT broker...")

client.connect(
    BROKER,
    PORT,
    60
)


# ============================================================
# 18. DISPLAY SYSTEM INFORMATION
# ============================================================

print("\n========================================")

print(
    "REAL-TIME ML + RAG + LLM + SQLITE"
)

print("========================================")

print(
    "ML Model:",
    MODEL_PATH
)

print(
    "Maintenance Manual:",
    PDF_PATH
)

print(
    "Groq Model:",
    LLM_MODEL
)

print(
    "Database:",
    DB_PATH
)

print(
    "MQTT Broker:",
    BROKER
)

print(
    "MQTT Topic:",
    TOPIC
)

print(
    "========================================"
)

print(
    "Waiting for live sensor data..."
)

print(
    "Press STOP / INTERRUPT to stop monitoring."
)

print(
    "========================================")


# ============================================================
# 19. START MQTT LOOP
# ============================================================

try:

    client.loop_forever()


except KeyboardInterrupt:

    print("\n")

    print("========================================")

    print(
        "MQTT monitoring stopped by user."
    )

    print("========================================")


finally:

    try:

        client.disconnect()

    except:

        pass


    print(
        "\nLatest SQLite database:"
    )

    print(DB_PATH)


    # ========================================================
    # DOWNLOAD SQLITE DATABASE
    # ========================================================

    try:

        if os.path.exists(DB_PATH):

            print(
                "\nDownloading SQLite database..."
            )

            files.download(DB_PATH)

            print(
                "SQLite database download started."
            )

        else:

            print(
                "SQLite database file not found."
            )

    except Exception as download_error:

        print(
            "Database download error:",
            download_error
        )


print("\n========================================")

print(
    "PROGRAM STOPPED SUCCESSFULLY"
)

print("========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!
Model path: /content/drive/MyDrive/temperature_humidity_model.pkl

Loading maintenance manual...
Maintenance manual loaded successfully!
Number of pages: 3
PDF split into 14 chunks

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
FAISS vector database created successfully!

Loading Groq API key...
Groq LLM initialized successfully!
Using Groq model: openai/gpt-oss-20b
SQLite database created successfully!

Connecting to MQTT broker...

REAL-TIME ML + RAG + LLM + SQLITE
ML Model: /content/drive/MyDrive/temperature_humidity_model.pkl
Maintenance Manual: /content/drive/MyDrive/temperature_humidity_maintenance_manual.pdf
Groq Model: openai/gpt-oss-20b
Database: temperature_monitoring.db
MQTT Broker: broker.hivemq.com
MQTT Topic: power/fault
Waiting for live sensor data...
Press STOP / INTERRUPT to stop monitoring.

Connected to MQTT broker
Subscribed to: power/fault

Received: {"Temperature":29.6,"Humidity":2.0}
ML Prediction: Fault
RAG search completed successfully!
Groq LLM recommendation generated successfully!


       REAL-TIME ML + RAG + LLM RESULT
Date              : 2026-09-21
Time              : 22:52:03
Temperature       : 29.6 °C
Humidity          : 2.0 %
ML Predictio

/tmp/ipykernel_5744/478342674.py:568: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat(



Received: {"Temperature":29.6,"Humidity":2.0}
ML Prediction: Fault
RAG search completed successfully!
Groq LLM recommendation generated successfully!


       REAL-TIME ML + RAG + LLM RESULT
Date              : 2026-09-21
Time              : 22:52:08
Temperature       : 29.6 °C
Humidity          : 2.0 %
ML Prediction     : Fault
Fault Description : Temperature and Humidity Abnormality Detected
----------------------------------------------------------------------
LLM MAINTENANCE RECOMMENDATION
----------------------------------------------------------------------
**Maintenance Recommendation**

| Item | Detail |
|------|--------|
| **Possible Cause** | The sensor is reporting a very low humidity (2 %) and a moderate temperature (29.6 °C). This could be due to: 1) a wiring or power issue that is corrupting the sensor signal, 2) the sensor operating outside its specified range (check the datasheet), or 3) a sensor that has drifted or failed. |
| **Recommended Inspection** | 1. **Power &

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

SQLite database download started.

PROGRAM STOPPED SUCCESSFULLY
